[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/smoke_rings/smoke_rings.ipynb)

# Smoke Rings

This example moves thin vortex filaments, such as the core of a smoke ring, through an ideal fluid. In Euler flow, the flow of a fluid without viscosity that cannot be compressed, vorticity gathered in a thin tube along a curve moves the fluid everywhere around it, by the Biot–Savart law, and the fluid carries the tube along. The largest part of that motion comes from close to each point of the curve: it moves the point across the curve, in the direction perpendicular both to the curve and to the way it bends, at a speed set by how sharply it bends. Keeping that part alone is the smoke ring flow, the local induction approximation: a circle flies along its axis without changing shape, any other curve writhes as it goes, and Hoffmann gives the flow for polygons with edges of unit length and calls the surface the curve sweeps over time its Hashimoto surface. Keeping the whole Biot–Savart law lets separate rings move each other: two smoke rings one behind the other take turns passing through each other, leapfrogging.

The filament's points are antivectors, intrinsic like all geometry, and so are its edges. The velocity of each vertex comes from the turn between its two edges, `outgoing * ~incoming`: its bivector over its scalar part, times the circulation, an intrinsic scalar, is the step along the binormal, scaled by how sharply the polygon bends there. The discrete time steps fold each offset about the diagonal of a quad, a point, which plus the transformation's parameter is a quaternion: no volume and no handedness enter.

In the vector notation of fluid dynamics the smoke ring flow reads as $\dot\gamma = \kappa\,\mathbf B = \gamma' \times \gamma''$, for a curve parametrized by arc length, the local induction approximation of the Biot–Savart law.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import VGA3D as ga
from examples.util.animation import save_figure, show_animation
from examples.fluids.smoke_rings import render
from examples.fluids.smoke_rings.core import closed, ring, torus_knot, wavy_ring

In [ ]:
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                # Scalar+
Vector = ga.gatype.vector()                                # Vector+
Bivector = ga.gatype.bivector()
Rotor = ga.gatype.rotor()                                  # Rotor+
# Points, and their differences, the edges and offsets: antivectors, intrinsic.
Point = ga.gatype.antivector()                             # Antivector-
# The filament's circulation changes sign under a mirror: an intrinsic scalar, set to one here.
Circulation = ga.gatype.antipseudoscalar()            # Antipseudoscalar-
circulation = mv(Circulation, [1.0])
# The Bäcklund transformation's parameter, added to the quad's diagonal, a point: an intrinsic scalar too.
Parameter = ga.gatype.antipseudoscalar()              # Antipseudoscalar-
# A polygon's vector area, the join of its points: an intrinsic vector, a patch.
Patch = Vector.derive.flipped_orientation

## 1. The turn at a vertex

At a vertex of a polygon, the outgoing edge times the reverse of the incoming one, `outgoing * ~incoming`, is the turn from one edge to the other. Its bivector over its scalar part is the tangent of half the turning angle times the turn's unit bivector, and times the circulation, an intrinsic scalar, that bivector is a point: the step along the binormal at the vertex. Twice the step, over the mean length of the two edges, is the vertex's velocity; on edges of unit length it is exactly Hoffmann's, and for short edges it tends to the curvature times the binormal. A regular polygon makes the same turn at every vertex, so every vertex moves alike: the ring flies along its axis without changing shape, at twice the tangent of half its turn per vertex.

In vector notation the velocity reads $\dot\gamma_k = 2\, S_k \times S_{k-1} / (1 + \langle S_k, S_{k-1}\rangle)$ for edges $S$ of unit length, and in the smooth limit $\dot\gamma = \kappa B$, the curvature times the Frenet binormal.

In [ ]:
# A regular polygon's vertices, its edges of unit length.
n_ring = 60

In [ ]:
def edges(points: Point) -> Point:
    """The edges of closed polygons `[..., n]`, edge k from point k to point k + 1."""
    return concatenate([points[..., 1:], points[..., :1]], axis=-1) - points     # [..., n] Point-


def velocity(points: Point, circulation: Circulation) -> Point:
    """The smoke ring flow's velocity at each vertex `[..., n]` of a filament of the given circulation:
    the turn's Cayley parameter, twice, times the circulation, over the mean edge length."""
    outgoing = edges(points)                                                     # [..., n] Point-
    incoming = concatenate([outgoing[..., -1:], outgoing[..., :-1]], axis=-1)    # [..., n] Point-
    lengths = outgoing.norm() * incoming.norm()                                  # [..., n] Scalar+
    # The commutator of the two edges is the bivector of the turn; over the lengths plus the edges paired,
    # it is the tangent of half the turning angle.
    binormal_step = 2 * incoming.commutator(outgoing) * circulation / (lengths + outgoing.dot(incoming))   # [..., n] Point-
    return binormal_step / ((outgoing.norm() + incoming.norm()) / 2)             # [..., n] Point-


flying = velocity(ring(n_ring), circulation)                                     # [n] Point-

In [ ]:
print("spread of the ring's velocities:", (flying - flying[0]).norm().to_array().max())
print("speed along the axis, and twice the tangent of the half turn:", (flying & mv.z).to_array()[0], 2 * np.tan(np.pi / n_ring))

## 2. The flow's invariants and the surface it sweeps

Each vertex moves perpendicular to both of its edges, so no edge changes its length, whatever the lengths are. The flow keeps the polygon's vector area too, the sum of each point joined with the next, which is the impulse of a vortex filament. A ring carrying a standing wave of three crests flies along its axis while the waves travel around it; joined by strips from one time to the next, the rings make the surface it sweeps, its Hashimoto surface, coloured from early to late.

In [ ]:
# The wavy ring's vertices, the waves around it and the tilt of its edges; the flow's step, its number
# of steps, and the steps between kept rings.
n_wavy, wavy_mode, wavy_tilt = 90, 3, 0.4
surface_step, n_surface_steps, surface_stride = 0.4, 750, 10

In [ ]:
def impulse(points: Point) -> Patch:
    """The vector area of closed polygons, twice over: the sum of each point joined with the next."""
    return (points & (points + edges(points))).sum(axis=-1)                      # [...] Patch-


def flow(points: Point, rate, step: float) -> Iterator[Point]:
    """A flow of polygons with the velocity `rate`, by classical Runge–Kutta steps: the polygons at every
    step, without end."""
    while True:
        yield points
        first = rate(points)
        second = rate(points + first * (step / 2))
        third = rate(points + second * (step / 2))
        fourth = rate(points + third * step)
        points = points + (first + 2 * second + 2 * third + fourth) * (step / 6)


def flying_rate(points: Point) -> Point:
    """The smoke ring flow's velocity at every vertex."""
    return velocity(points, circulation)


wavy = wavy_ring(n_wavy, wavy_mode, wavy_tilt)                                   # [n] Point-
surface = stack(list(islice(flow(wavy, flying_rate, surface_step), 0, n_surface_steps + 1, surface_stride)))   # [times, n] Point-
lengths = edges(surface).norm().to_array()                                       # [times, n]
areas = impulse(surface)                                                         # [times] Point-

In [ ]:
print("largest change of an edge's length:", np.abs(lengths - lengths[0]).max())
print("largest change of the impulse, relative:", ((areas - areas[0]).norm() / areas[0].norm()).to_array().max())
save_figure(render.draw_surface(closed(surface)), "smoke_rings_surface");

## 3. A knotted vortex

A vortex filament tied in a trefoil knot, on a torus about the axis, turns about the axis as it flies along it, and keeps its knotted shape while its loops ripple: the flow moves each part of the knot by its own bending, and the trefoil's three loops bend alike. The knot's edges are shorter than one and not all equal, so its step is shorter, and its velocity, divided by the mean edge length, is the curvature times the binormal rather than Hoffmann's unit-edge form.

In [ ]:
# The trefoil's windings around the axis and around the tube, the tube's radii and its vertices; the
# flow's step, its number of steps, the steps between frames, and the duration of a frame.
knot_windings, knot_radii, n_knot = (2, 3), (6.0, 2.5), 240
knot_step, n_knot_steps, knot_stride = 1 / 30, 3600, 40
duration_ms = 40

In [ ]:
knot = torus_knot(*knot_windings, *knot_radii, n_knot)                          # [n] Point-
curves = stack(list(islice(flow(knot, flying_rate, knot_step), 0, n_knot_steps + 1, knot_stride)))   # [times, n] Point-
knot_lengths = edges(curves).norm().to_array()                                  # [times, n]

In [ ]:
print("largest change of an edge's length:", np.abs(knot_lengths - knot_lengths[0]).max())
show_animation(render.animate_curves(closed(curves), closed(flying_rate(curves).norm())), "smoke_rings_knot", duration_ms)

## 4. Discrete time

Time can be made discrete as well, by Hoffmann's Bäcklund transformation: the new polygon is the old one moved by an offset of one fixed length at every vertex, with every new edge again of unit length. Each offset follows from the one before by a fold about the diagonal of the quad between the old edge and the new one, `edge - offset`, a point. Plus the transformation's parameter, an intrinsic scalar like the circulation, the diagonal is a quaternion, and the fold is its sandwich, `((edge - offset) + twist).normalized() >> offset`; it keeps each new edge's part along the diagonal, so every new edge is exactly as long as the old.

For a closed polygon the offsets must come round to where they started. On the sphere of unit offsets, a fold across an edge acts as a Lorentz transformation, a boost along the edge and a turn about it, so a lap around the polygon is a product of rotors, and minus the last edge, carried once around, lands on the lap's attracting fixed point. With unit offsets the new polygon is the old one moved by the smoke ring flow over about half the parameter, relabelled by one vertex, so a transformation is a step in time: every edge keeps exactly unit length, and the ring, a discrete elastic curve, moves exactly rigidly. The wavy ring stepped this way sweeps the discrete Hashimoto surface, every quad folded about its diagonal.

In quaternion notation, with the vectors as imaginary quaternions, the fold is the Riccati-type equation $(\lambda + \tilde S)(\lambda + \operatorname{Re}\nu + v) = (\lambda + \operatorname{Re}\nu + v_{+})(\lambda + S)$ for all $\lambda$, with $\tilde S$ the new edge and $v$, $v_{+}$ the offsets at its ends, and the Lorentz transformation the matrix of $SL(2, \mathbb{C})$ that acts on the sphere by the Möbius map.

In [ ]:
# The transformation's parameter, an intrinsic scalar; the rings' vertices, and the steps.
twist = mv(Parameter, [0.5])                                                     # [] Parameter-
n_discrete, n_discrete_steps = 40, 100

In [ ]:
def fold(edge: Point, offset: Point, twist: Parameter) -> Point:
    """The next offset of a Bäcklund transformation across an edge: the current one turned about the
    diagonal `edge - offset` of the quad between them, by the quaternion the diagonal and the parameter
    make together."""
    return ((edge - offset) + twist).normalized() >> offset                      # [...] Point-


def backlund(points: Point, start: Point, twist: Parameter) -> Point:
    """One Bäcklund transformation of a polygon `[..., n]` with the given parameter, from the offset at its
    first vertex: every later offset is the one before, folded across the edge between them."""
    sides = points[..., 1:] - points[..., :-1]                                   # [..., n - 1] Point-

    # The offsets are folded one after another: a product of the folds' Lorentz rotors grows from edge to
    # edge until round-off swamps the offsets.
    def offsets() -> Iterator[Point]:
        offset = start
        yield offset
        for index in range(sides.shape[-1]):
            offset = fold(sides[..., index], offset, twist)
            yield offset

    return points + stack(list(offsets()), axis=-1)                              # [..., n] Point-


def periodic(points: Point, twist: Parameter) -> Point:
    """The unit offset at the first vertex of closed polygons of unit edges `[..., n]` that comes round to
    itself after a lap: minus the last edge, carried once around by the lap's Lorentz transformation,
    which lands it on the lap's attracting fixed point."""
    sides = edges(points)                                                        # [..., n] Point-
    # Every fold is a boost along its edge and a turn about it, its rapidity and the turn's angle the real
    # and imaginary parts of one logarithm, of 1 - 2 i / twist.
    rapidity = (1 + 4 / twist.squared()).log() / 2                               # [] Scalar+
    angle = -(2 * twist.inverse()).arctan()                                      # [] Parameter-
    folds = (-(rapidity * sides.dual_inverse() + angle * sides) / 2).exp()       # [..., n] Multivector+
    lap = folds.cumprod(axis=-1)[..., -1]                                        # [...] Multivector+
    # In the algebra of space a Lorentz transformation acts on paravectors by the sandwich with the reverse;
    # the null paravector of an offset is one plus its direction, carried once around.
    image = lap * (1 - sides[..., -1].dual_inverse()) * lap.reverse()            # [...] Multivector+
    # The paravector's bivector and volume parts are zero.
    return (image.cast(Vector) / image.cast(Scalar)).dual()                      # [...] Point-


def advance(points: Point, twist: Parameter) -> Point:
    """One step of the discrete-time smoke ring flow of closed polygons of unit edges `[..., n]`: the
    Bäcklund transformation with periodic offsets, relabelled by one vertex, a time of about half the
    parameter along the flow."""
    return backlund(closed(points), periodic(points, twist), twist)[..., 1:]     # [..., n] Point-


# The fold, and the Lorentz transformation of the same edge, on a few unit offsets.
edge = mv.z.dual()                                                               # [] Point-
offsets = mv.antivector([[1, 0, 0], [0, 1, 0], [0, -0.6, 0.8]])                  # [3] Point-
rapidity, angle = (1 + 4 / twist.squared()).log() / 2, -(2 * twist.inverse()).arctan()
rotor = (-(rapidity * edge.dual_inverse() + angle * edge) / 2).exp()
image = rotor * (1 + offsets.dual_inverse()) * rotor.reverse()
lorentz = (image.cast(Vector) / image.cast(Scalar)).dual()                       # [3] Point-

# The plain ring and the wavy one, stepped together.
rings = stack([ring(n_discrete), wavy_ring(n_discrete, wavy_mode, wavy_tilt)])   # [rings, n] Point-


def stepped() -> Iterator[Point]:
    curves = rings
    while True:
        yield curves
        curves = advance(curves, twist)


discrete = stack(list(islice(stepped(), n_discrete_steps + 1)))                  # [steps + 1, rings, n] Point-
discrete_lengths = edges(discrete).norm().to_array()                             # [steps + 1, rings, n]
moved_ring = discrete[-1, 0]                                                     # [n] Point-

In [ ]:
print("largest difference between the fold and the Lorentz transformation:", (lorentz - fold(edge, offsets, twist)).norm().to_array().max())
print("largest departure of an edge from unit length:", np.abs(discrete_lengths - 1).max())
print("spread of the ring's heights after the steps:", np.ptp((moved_ring & mv.z).to_array()))
save_figure(render.draw_surface(closed(discrete[:, 1])), "smoke_rings_discrete");

## 5. A loop running along a straight vortex

A soliton is a solitary wave that travels without changing its shape. On a straight vortex line it is a loop that runs along the line and turns about it as it goes, Hasimoto's soliton, and one Bäcklund transformation of a straight line makes one: an offset leaning just off the forward direction stays near it for a while, then crosses over to the backward one, and the new polygon carries a loop where it crosses. The offset's length sets the loop's size, and the parameter how drawn out it is along the line.

The steps of section 4 carry the loop along, the line running on straight past its ends. Through the loop the tangent turns once around: the angles of the turns at the vertices, `outgoing * ~incoming`, add up to a full turn at every step, whatever the loop's size. The animation draws the line at each step and, behind it, the screw-shaped tube the loop sweeps; the figure after it draws the turning angles along the line and through the steps, a streak of one width moving at one speed.

In the notation of the theory of curves, Hasimoto's map $\psi = \kappa\, e^{i\int\tau\, ds}$ of the curvature $\kappa$ and the torsion $\tau$ takes the smoke ring flow to the nonlinear Schrödinger equation $i\psi_t + \psi_{ss} + \tfrac{1}{2}|\psi|^2\psi = 0$, and the soliton's curvature is $\kappa = 2\nu\operatorname{sech}(\nu s)$, whose integral is $2\pi$ for every $\nu$.

In [ ]:
# A straight line of unit edges; how far the first offset leans off the forward direction, its length and
# the parameter; the parameter of the steps that carry the loop, their number, one a frame, and the duration of a
# frame.
n_line, lean, offset_length = 75, 1e-5, 4.0
loop_twist, carry_twist = mv(Parameter, [2.0]), mv(Parameter, [1.0])           # [] Parameter-
n_carry_steps, carry_duration_ms = 90, 40

In [ ]:
def turns(points: Point) -> Rotor:
    """The turn at every inner vertex of polygons of unit edges `[..., n]`: the rotor from the incoming
    edge to the outgoing one, the square of the rotor that carries one onto the other."""
    sides = points[..., 1:] - points[..., :-1]                                  # [..., n - 1] Point-
    return sides[..., 1:] * ~sides[..., :-1]                                    # [..., n - 2] Rotor+


line = mv.x.dual() * (np.arange(n_line) - n_line / 2)                           # [n] Point-
start = ((mv.zx * lean).exp() >> mv.x.dual()) * offset_length                   # [] Point-


def carried() -> Iterator[Point]:
    curve = backlund(line, start, loop_twist)                                   # [n] Point-
    while True:
        yield curve
        # Offsets from minus the first edge, relabelled by one vertex; the line runs on straight past its end.
        moved = backlund(curve, curve[0] - curve[1], carry_twist)[1:]           # [n - 1] Point-
        curve = concatenate([moved, 2 * moved[-1:] - moved[-2:-1]])             # [n] Point-


solitons = stack(list(islice(carried(), n_carry_steps + 1)))                    # [steps + 1, n] Point-
angles = turns(solitons).log().norm().to_array()                                # [steps + 1, n - 2]

In [ ]:
print("total turning through the loop at the first and the last step, in full turns:", angles[[0, -1]].sum(axis=-1) / (2 * np.pi))
# The flow's speed along the line; at its two ends, which the closing edge would bend, their neighbours'.
inner = flying_rate(solitons).norm()[..., 1:-1]                 # [steps, n - 2] Scalar+
show_animation(render.animate_trail(solitons, concatenate([inner[..., :1], inner, inner[..., -1:]], axis=-1)),
               "smoke_rings_soliton", carry_duration_ms)
save_figure(render.draw_turns(turns(solitons)), "smoke_rings_turns");

## 6. Two rings leapfrogging

The whole Biot–Savart law moves every vertex by every segment of every filament, not by its own two edges alone: a straight segment moves a point by the commutator of the point's separation from the segment's middle with the segment, over the cube of the separation's length, softened by a core radius so that the law stays finite on the filament itself, the vortex filament of Rosenhead and Moore. Two equal rings, one a little behind the other on the same axis, move each other: the front ring widens and slows, the rear one narrows, speeds up and passes through the front one, and the two change places, twice in every cycle. The figure draws the tube each ring sweeps through two cycles in its own colour, the half nearer the viewer cut away: wherever one ring passes through the other, its tube runs inside.

In the notation of fluid dynamics the law reads $u(x) = \frac{\Gamma}{4\pi} \oint \frac{d\ell \times (x - y)}{(|x - y|^2 + \delta^2)^{3/2}}$, for a filament of circulation $\Gamma$ and core radius $\delta$.

In [ ]:
# Two equal rings on one axis, one behind the other: their radius, vertices and heights, their circulations
# and core radius; the flow's step and number of steps, through two cycles, and the steps between frames.
radius, n_frog, frog_heights = 5.0, 48, np.array([0.0, 2.5])
circulations, core_radius = mv(Circulation, [[10.0], [10.0]]), 0.4
frog_step, n_frog_steps, frog_stride = 0.4 / 3, 150, 2

In [ ]:
def induced(points: Point, circulations: Circulation, core_radius: float) -> Point:
    """The velocity at every vertex of closed polygonal vortex filaments `[rings, n]`, of the given
    circulations `[rings]`, induced by every segment of every filament: the commutator of the vertex's
    separation from the segment's middle with the segment, over the softened cube of the separation."""
    segments = edges(points)                                                     # [rings, n] Point-
    separations = points.reshape(-1)[:, None] - (points + segments / 2).reshape(-1)[None, :]   # [rings n, rings n] Point-
    softened = separations.norm_squared() + core_radius**2                       # [rings n, rings n] Scalar+
    strengths = circulations[..., None].broadcast_to(points.shape).reshape(-1) / (4 * np.pi)   # [rings n] Circulation
    pulls = separations.commutator(segments.reshape(-1)) * strengths / (softened * softened.square_root())
    return pulls.sum(axis=1).reshape(points.shape)                               # [rings, n] Point-


rings = (ring(n_frog) * (radius * 2 * np.sin(np.pi / n_frog)))[None, :] + mv.z.dual() * frog_heights[:, None]   # [rings, n] Point-
frogs = stack(list(islice(flow(rings, lambda points: induced(points, circulations, core_radius), frog_step), n_frog_steps + 1)))   # [times, rings, n] Point-
# Each point paired with the axis: its height along it.
heights = (frogs & mv.z).to_array().mean(axis=-1)                                # [times, rings]
radii = (frogs - mv.z.dual() * (frogs & mv.z)).norm().to_array().mean(axis=-1)   # [times, rings]
passes = np.flatnonzero(np.diff(np.sign(heights[:, 1] - heights[:, 0])))

In [ ]:
print("times at which one ring passes through the other:", passes * frog_step)
save_figure(render.draw_swept(closed(frogs)), "smoke_rings_swept");

In [ ]:
# Each time's rings pushed by both: the induced flow couples every vertex of a time, and times are a batch.
speeds = stack([induced(frame, circulations, core_radius) for frame in frogs]).norm()   # [times, rings, n] Scalar+
show_animation(render.animate_curves(closed(frogs[::frog_stride]), closed(speeds[::frog_stride])), "smoke_rings_leapfrog",
               duration_ms)

In [ ]:
# checks
# The ring flies rigidly along its axis at twice the tangent of half its turn per vertex.
np.testing.assert_allclose((flying - flying[0]).norm().to_array(), 0.0, atol=1e-13)
np.testing.assert_allclose(np.abs((flying & mv.z).to_array()), 2 * np.tan(np.pi / n_ring), rtol=1e-12)
# Both flows keep every edge's length; the wavy ring keeps its impulse.
np.testing.assert_allclose(lengths, np.broadcast_to(lengths[0], lengths.shape), atol=1e-9)
np.testing.assert_allclose(knot_lengths, np.broadcast_to(knot_lengths[0], knot_lengths.shape), atol=1e-9)
np.testing.assert_allclose(((areas - areas[0]).norm() / areas[0].norm()).to_array(), 0.0, atol=1e-6)
# In discrete time every edge keeps unit length, and the ring moves rigidly.
np.testing.assert_allclose(discrete_lengths, 1.0, atol=1e-12)
np.testing.assert_allclose((moved_ring & mv.z).to_array(), (moved_ring[0] & mv.z).to_array(), atol=1e-12)
# The soliton keeps unit edges, and the tangent turns once around through the loop at every step; the
# tolerance allows for the loop's tails beyond the line's ends.
np.testing.assert_allclose((solitons[:, 1:] - solitons[:, :-1]).norm().to_array(), 1.0, atol=1e-12)
np.testing.assert_allclose(angles.sum(axis=-1), 2 * np.pi, atol=1e-3)
# The leapfrogging rings take turns, the narrower one passing through the wider.
behind = np.argmin(heights[passes], axis=-1)
assert len(passes) >= 4 and np.all(radii[passes, behind] < radii[passes, 1 - behind])